# Tamil Hierarchical Hate Speech Classifier (5 levels)
**MuRIL** backbone · **NLLB-200 back-translation** (train-only, targeted at implicit & minority hate) · **out-of-fold hard-negative mining** (stage-2 re-weighting) · **F2-optimised thresholds** (tuned on validation) · **hierarchical heads + constrained decoding**

Pipeline:
1. Clean data & labels → per-level label masks (missing sub-labels are ignored, not trained as "Nil")
2. Stratified 70/15/15 split (before any augmentation → no leakage)
3. NLLB-200 back-translation of training rows (Tamil → English/Hindi → Tamil), quality-filtered, cached
4. Stage 1: train hierarchical multi-task MuRIL (class-weighted focal loss, differential LR, fp16, early stopping)
5. Out-of-fold hard-negative mining (3 fold models score the rows they never saw) → Stage 2 fine-tuning with hard examples up-weighted; likely label errors exported to `label_review.csv`
6. F2-optimised L1 threshold + per-class F2 offsets for L2–L5 (tuned on validation only)
7. Test evaluation: oracle-gated and end-to-end metrics, implicit vs explicit recall, ablation table
8. Save model + thresholds, `predict()` for inference

Level 4 head predicts a shared **abuse type** (Verbal Abusive, Slut-shaming, …); the full subcategory is rebuilt from target + explicit/implicit + type.

Runtime on a Colab T4: ~10–20 min back-translation (first run only; upload `bt_cache.csv` to skip) + ~35 min training (stage 1, 3 OOF fold models, stage 2).

In [ ]:
!pip -q install -U transformers accelerate sentencepiece

In [ ]:
import os, re, gc, json, time, random, shutil, unicodedata, difflib
from collections import Counter
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import transformers
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, fbeta_score, recall_score, precision_score,
                             classification_report, confusion_matrix, accuracy_score)
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
transformers.logging.set_verbosity_error()

CFG = dict(
    seed=42,
    model_name='google/muril-base-cased',
    max_len=128,
    batch_size=16, eval_batch_size=64,
    # stage 1
    stage1_epochs=10, enc_lr=3e-5, head_lr=5e-4,
    # stage 2 (hard-negative mining)
    stage2_epochs=4, stage2_enc_lr=1e-5, stage2_head_lr=2e-4,
    weight_decay=0.01, warmup_ratio=0.1, patience=3, head_dropout=0.2,
    # labels / loss
    min_class_count=15,                 # rare classes merged *within their parent* (e.g. ImplicitLGBTQ-Other)
    loss_weights={1: 1.0, 2: 0.8, 3: 0.8, 4: 0.6, 5: 0.5},
    cw_power=0.5,                       # class weight = (inverse frequency)^0.5
    focal_gamma=1.0, label_smoothing=0.05,
    # NLLB-200 back-translation
    use_bt=True,
    bt_model='facebook/nllb-200-distilled-600M',
    bt_batch=32, bt_beams=4, bt_max_tokens=160, bt_max_chars=400,
    bt_pivots_common=['eng_Latn'],               # 1 paraphrase for ordinary hate rows
    bt_pivots_rare=['eng_Latn', 'hin_Deva'],     # 2 paraphrases for implicit / minority-class rows
    bt_rare_threshold=60,                        # a row is "minority" if any of its classes has < 60 train rows
    bt_nonhate_ratio=0.35,                       # also paraphrase some NON-HATE so "translationese" != hate
    bt_min_tamil_ratio=0.5, bt_sim_range=(0.15, 0.92), bt_len_ratio=(0.5, 2.0),
    bt_cache='bt_cache.csv',
    # hard-negative mining
    use_hnm=True, hnm_top_frac=0.15, hnm_weight=2.0,
    hnm_folds=3, hnm_fold_epochs=3,     # out-of-fold mining: each training row is scored by a model that never saw it
    hnm_drop_suspects=False,            # True = drop very confident OOF disagreements (likely label errors) from stage 2
    # thresholds
    use_thresholds=True, beta=2.0,
    out_dir='tamil_hate_hier_model',
)
SEED = CFG['seed']

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('device:', DEVICE, '| torch', torch.__version__, '| transformers', transformers.__version__)
if DEVICE.type != 'cuda':
    print('WARNING: no GPU - set Runtime > Change runtime type > T4 GPU. Back-translation will be skipped.')
    CFG['use_bt'] = False

device: cuda | torch 2.14.1+cu130 | transformers 5.22.2


In [ ]:
# ---------- load data ----------
from google.colab import files
print('Select your dataset CSV (e.g. train.csv). You can also select bt_cache.csv from a previous run '
      'at the same time to skip back-translation.')
up = files.upload()
data_files = [k for k in up if not k.startswith('bt_cache')]
assert data_files, 'No dataset CSV uploaded - run this cell again and choose train.csv.'
CSV_PATH = data_files[0]
for k in up:
    if k.startswith('bt_cache'):
        if k != CFG['bt_cache']:
            os.replace(k, CFG['bt_cache'])        # Colab renames re-uploads to "bt_cache (1).csv"
        print('bt_cache uploaded -> back-translation will be loaded from cache')
print('uploaded:', CSV_PATH)
raw = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(raw.shape, raw.columns.tolist())

Select your dataset CSV (e.g. train.csv). You can also select bt_cache.csv from a previous run at the same time to skip back-translation.


Saving bt_cache.csv to bt_cache (1).csv
Saving train.csv to train.csv
bt_cache uploaded -> back-translation will be loaded from cache
uploaded: train.csv
(4255, 7) ['Input_ID', 'Input_Text', 'Level1_Class', 'Level2_Target', 'Level3_Category', 'Level4_Subcategory', 'Level5_Bias']


In [ ]:
# ---------- text + label cleaning ----------
LEVELS = ['Level1_Class', 'Level2_Target', 'Level3_Category', 'Level4_Subcategory', 'Level5_Bias']
COLS = {i + 1: c for i, c in enumerate(LEVELS)}
URL_RE = re.compile(r'https?://\S+|www\.\S+')

def normalize_text(s):
    s = unicodedata.normalize('NFC', str(s))
    s = URL_RE.sub(' ', s)
    s = re.sub(r'@\w+', '@user', s)
    s = re.sub(r'(.)\1{3,}', r'\1\1\1', s)          # "சூப்பர்ர்ர்ர்ர்" / "!!!!!!" -> max 3 repeats
    return re.sub(r'\s+', ' ', s).strip()

df = raw[['Input_Text'] + LEVELS].copy()
df = df.dropna(subset=['Input_Text', 'Level1_Class'])
for c in LEVELS:
    df[c] = df[c].fillna('Nil').astype(str).str.strip()
df['Input_Text'] = df['Input_Text'].map(normalize_text)
df['Level1_Class'] = df['Level1_Class'].str.upper()
df['Level3_Category'] = df['Level3_Category'].str.upper().replace({'NIL': 'Nil'})
n_raw = len(df)

df = df[(df.Input_Text.str.len() > 0) & df.Level1_Class.isin(['HATE', 'NON-HATE'])]   # drops the 9 'NIL' rows
conflict = df.groupby('Input_Text').Level1_Class.nunique()
df = df[~df.Input_Text.isin(conflict[conflict > 1].index)]                             # same text, different L1
df = df.drop_duplicates('Input_Text').reset_index(drop=True)
print(f'rows: {n_raw} -> {len(df)} after removing NIL / conflicting / duplicate texts')

# label normalisation
df['Level2_Target'] = df['Level2_Target'].replace({'Men- no profane': 'Hate against Men'})
df['Level5_Bias'] = df['Level5_Bias'].replace({'Religional': 'Religion', 'Regional': 'Geography', 'Nil': 'No-Bias'})

is_hate = df.Level1_Class.eq('HATE')
MIN = CFG['min_class_count']

# Level4: 'Nil' is a real answer ("None") for untargeted profanity; for targeted hate it is a missing annotation
df.loc[is_hate & df.Level4_Subcategory.eq('Nil') & df.Level2_Target.eq('No Target- Profane'), 'Level4_Subcategory'] = 'None'

# merge rare Level4 classes inside their parent (Explicit/Implicit x Target), not into one global 'Other'
SUB_RE = re.compile(r'^(Explicit|Implicit)(Men|Women|LGBTQ)-(.+)$')
vc4 = df.loc[is_hate, 'Level4_Subcategory'].value_counts()
def merge_l4(x):
    if x in ('Nil', 'None', 'Sarcasm') or vc4.get(x, 0) >= MIN:
        return x
    m = SUB_RE.match(x)
    return f'{m.group(1)}{m.group(2)}-Other' if m else 'Other'
df.loc[is_hate, 'Level4_Subcategory'] = df.loc[is_hate, 'Level4_Subcategory'].map(merge_l4)

def merge_rare(col, keep=('Nil',)):
    vc = df.loc[is_hate, col].value_counts()
    df.loc[is_hate, col] = df.loc[is_hate, col].map(lambda x: x if (x in keep or vc.get(x, 0) >= MIN) else 'Other')
merge_rare('Level2_Target'); merge_rare('Level4_Subcategory'); merge_rare('Level5_Bias', keep=('No-Bias',))

# values that mean "not annotated" -> ignored in the loss/metrics for that level (-100)
IGNORE = {2: {'Nil'}, 3: {'Nil'}, 4: {'Nil'}, 5: set()}
CLASSES = {1: ['NON-HATE', 'HATE']}
for k in (2, 3, 5):
    CLASSES[k] = sorted(set(df.loc[is_hate, COLS[k]]) - IGNORE[k])

# ---- Level 4: the head predicts a shared ABUSE TYPE; the full subcategory is rebuilt as
#      (Explicit/Implicit from L3) + (target from L2) + type.  e.g. Implicit + Women + Slut-shaming
#      -> 'ImplicitWomen-Slut-shaming'. Types are shared across targets, so each class gets more data.
FULL4 = sorted(set(df.loc[is_hate, 'Level4_Subcategory']) - IGNORE[4])   # full subcategory labels (evaluation)
TARGET_NAME = {'Men': 'Hate against Men', 'Women': 'Hate against Women', 'LGBTQ': 'Hate against LGBTQ+'}
def split_full(f):
    m = SUB_RE.match(f)
    if not m:
        return None, None, f                                  # Sarcasm / None / Other: valid for any target
    return m.group(1).upper(), TARGET_NAME[m.group(2)], m.group(3)
CLASSES[4] = sorted({split_full(f)[2] for f in FULL4})        # abuse types (what the L4 head predicts)
NUM = {k: len(v) for k, v in CLASSES.items()}
NUM_FULL4 = len(FULL4)
TYPE_OF = np.array([CLASSES[4].index(split_full(f)[2]) for f in FULL4])
T2_OF = np.array([CLASSES[2].index(split_full(f)[1]) if split_full(f)[1] in CLASSES[2] else -1 for f in FULL4])
T3_OF = np.array([CLASSES[3].index(split_full(f)[0]) if split_full(f)[0] in CLASSES[3] else -1 for f in FULL4])

def encode_labels(d):
    d = d.copy()
    h = d.Level1_Class.eq('HATE').values
    d['y1'] = h.astype(int)
    for k in (2, 3, 5):
        idx = {c: i for i, c in enumerate(CLASSES[k])}
        d[f'y{k}'] = [idx[v] if (hh and v in idx) else -100 for hh, v in zip(h, d[COLS[k]])]
    fidx = {c: i for i, c in enumerate(FULL4)}
    d['y4f'] = [fidx[v] if (hh and v in fidx) else -100 for hh, v in zip(h, d[COLS[4]])]
    d['y4'] = [int(TYPE_OF[v]) if v >= 0 else -100 for v in d['y4f']]
    return d

df = encode_labels(df)
print('num classes:', NUM, '| full L4 subcategories:', NUM_FULL4)
for k in range(2, 6):
    lab = df[f'y{k}'] >= 0
    print(f'\n{COLS[k]}{" (head = abuse type)" if k == 4 else ""}: {lab.sum()} labelled HATE rows, '
          f'{(df.y1.eq(1) & ~lab).sum()} unlabelled (ignored)')
    vals = df.loc[lab, 'y4'].map(lambda i: CLASSES[4][i]) if k == 4 else df.loc[lab, COLS[k]]
    print(vals.value_counts().to_string())

rows: 4255 -> 4240 after removing NIL / conflicting / duplicate texts
num classes: {1: 2, 2: 4, 3: 2, 5: 6, 4: 16} | full L4 subcategories: 24

Level2_Target: 2182 labelled HATE rows, 46 unlabelled (ignored)
Level2_Target
Hate against Women     1018
Hate against Men        610
No Target- Profane      411
Hate against LGBTQ+     143

Level3_Category: 2203 labelled HATE rows, 25 unlabelled (ignored)
Level3_Category
IMPLICIT    1242
EXPLICIT     961

Level4_Subcategory (head = abuse type): 2061 labelled HATE rows, 167 unlabelled (ignored)
y4
Verbal Abusive             578
None                       306
Sarcasm                    286
Character assassination    238
Slut-shaming               115
Other                       84
Dismissal                   79
Body shaming                76
Victim Blaming              70
Moral policing              56
Sexual Objectification      43
Masculinity Attacks         43
Steriotype                  27
Sexual Threat               23
Mocking Identity     

In [ ]:
# ---------- stratified split BEFORE augmentation ----------
df['strat'] = np.where(df.y1 == 1, 'H_' + df.Level2_Target, 'NH')
train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df.strat, random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df.strat, random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True).copy() for d in (train_df, val_df, test_df)]
for d in (train_df, val_df, test_df):
    d['is_aug'] = 0; d['sw'] = 1.0; d['src_text'] = None
print('train/val/test:', len(train_df), len(val_df), len(test_df))

dist = {}
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    for lvl, col, names in [('L1', 'y1', CLASSES[1]), ('L2', 'y2', CLASSES[2]), ('L3', 'y3', CLASSES[3]),
                            ('L4type', 'y4', CLASSES[4]), ('L4full', 'y4f', FULL4), ('L5', 'y5', CLASSES[5])]:
        y = d[col]
        for c, n in y[y >= 0].value_counts().items():
            dist[(lvl, names[c])] = dist.get((lvl, names[c]), {}) | {name: n}
print(pd.DataFrame(dist).T.fillna(0).astype(int).to_string())

# hierarchy compatibility: which full L4 subcategories occur with each (L2 target, L3 category) pair in TRAIN
COMPAT = np.zeros((NUM[2], NUM[3], NUM_FULL4), dtype=bool)
_t = train_df[(train_df.y2 >= 0) & (train_df.y3 >= 0) & (train_df.y4f >= 0)]
COMPAT[_t.y2.values, _t.y3.values, _t.y4f.values] = True
print('\nallowed (target, category) -> subcategory pairs:', int(COMPAT.sum()))

train/val/test: 2968 636 636
                                              train  val  test
L1     HATE                                    1560  334   334
       NON-HATE                                1408  302   302
L2     Hate against Women                       713  152   153
       Hate against Men                         427   91    92
       No Target- Profane                       288   62    61
       Hate against LGBTQ+                      100   22    21
L3     IMPLICIT                                 883  184   175
       EXPLICIT                                 661  145   155
L4type Verbal Abusive                           399   99    80
       Sarcasm                                  217   38    31
       None                                     211   47    48
       Character assassination                  163   33    42
       Slut-shaming                              85   14    16
       Dismissal                                 58   12     9
       Body shaming       

## NLLB-200 back-translation (training split only)
Hate rows that are **implicit** or belong to a **minority class** (L2/L4/L5 class with < `bt_rare_threshold` train rows) get two paraphrases (English and Hindi pivots); other hate rows get one. A proportional sample of NON-HATE rows is paraphrased too, so the classifier cannot learn "NLLB-style Tamil ⇒ hate". Outputs are filtered for duplicates, length drift, script and similarity, then cached to `bt_cache.csv`.

In [ ]:
def tamil_ratio(s):
    letters = [ch for ch in str(s) if ch.isalpha()]
    if not letters:
        return 0.0
    return sum('஀' <= ch <= '௿' for ch in letters) / len(letters)

_bt_toks = {}
def translate(texts, src, tgt, mdl, name):
    if src not in _bt_toks:
        _bt_toks[src] = AutoTokenizer.from_pretrained(name, src_lang=src)
    tok = _bt_toks[src]
    tgt_id = tok.convert_tokens_to_ids(tgt)
    out = [None] * len(texts)
    order = np.argsort([len(t) for t in texts])[::-1]          # length-sorted batches = less padding
    bs = CFG['bt_batch']
    for i in tqdm(range(0, len(order), bs), desc=f'{src}->{tgt}', leave=False):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], return_tensors='pt', padding=True, truncation=True,
                  max_length=CFG['bt_max_tokens']).to(DEVICE)
        with torch.inference_mode():
            gen = mdl.generate(**enc, forced_bos_token_id=tgt_id, num_beams=CFG['bt_beams'],
                               max_new_tokens=CFG['bt_max_tokens'])
        for j, s in zip(idx, tok.batch_decode(gen, skip_special_tokens=True)):
            out[j] = s
    return out

def build_bt_jobs(tr):
    rcols = ['y2', 'y3', 'y4f', 'y5']             # rarity measured on the full L4 subcategory
    freq = {c: tr.loc[tr[c] >= 0, c].value_counts().to_dict() for c in rcols}
    def rarity(r):
        vals = [freq[c].get(r[c], 10**9) for c in rcols
                if r[c] >= 0 and not (c == 'y5' and CLASSES[5][r.y5] == 'No-Bias')]
        return min(vals) if vals else 10**9
    ok = (tr.Input_Text.map(tamil_ratio) >= CFG['bt_min_tamil_ratio']) & (tr.Input_Text.str.len() <= CFG['bt_max_chars'])
    jobs = []
    for i, r in tr[(tr.y1 == 1) & ok].iterrows():
        rare = r.Level3_Category == 'IMPLICIT' or rarity(r) < CFG['bt_rare_threshold']
        for p in (CFG['bt_pivots_rare'] if rare else CFG['bt_pivots_common']):
            jobs.append((i, p))
    nh = tr[(tr.y1 == 0) & ok]
    nh = nh.sample(min(int(len(jobs) * CFG['bt_nonhate_ratio']), len(nh)), random_state=SEED)
    jobs += [(i, 'eng_Latn') for i in nh.index]
    return jobs

def back_translate(tr, forbidden_texts):
    jobs = build_bt_jobs(tr)
    print(f'back-translation jobs: {len(jobs)}  ({Counter(p for _, p in jobs)})')
    mdl = AutoModelForSeq2SeqLM.from_pretrained(CFG['bt_model']).half().to(DEVICE).eval()
    results = []
    for p in sorted({p for _, p in jobs}):
        idx = [i for i, pp in jobs if pp == p]
        src = [tr.at[i, 'Input_Text'] for i in idx]
        mid = translate(src, 'tam_Taml', p, mdl, CFG['bt_model'])
        back = translate(mid, p, 'tam_Taml', mdl, CFG['bt_model'])
        results += list(zip(idx, [p] * len(idx), mid, back))
    del mdl; _bt_toks.clear(); gc.collect(); torch.cuda.empty_cache()

    seen, rows, stats = set(forbidden_texts), [], Counter()
    lo, hi = CFG['bt_len_ratio']; slo, shi = CFG['bt_sim_range']
    for i, p, mid, bt in results:
        src = tr.at[i, 'Input_Text']; bt = normalize_text(bt or '')
        lr = len(bt) / max(len(src), 1)
        sim = difflib.SequenceMatcher(None, src, bt).ratio()
        if not bt or bt in seen:                          stats['duplicate/empty'] += 1; continue
        if not (lo <= lr <= hi):                          stats['length drift'] += 1; continue
        if tamil_ratio(bt) < CFG['bt_min_tamil_ratio']:   stats['not Tamil'] += 1; continue
        if not (slo <= sim <= shi):                       stats['too similar/too different'] += 1; continue
        seen.add(bt)
        row = tr.loc[i].copy()
        row['Input_Text'], row['is_aug'], row['src_text'], row['pivot'], row['pivot_text'] = bt, 1, src, p, mid
        rows.append(row); stats['kept'] += 1
    print('filter stats:', dict(stats))
    return pd.DataFrame(rows).reset_index(drop=True)

aug_df = pd.DataFrame(columns=train_df.columns)
if CFG['use_bt']:
    all_texts = set(train_df.Input_Text) | set(val_df.Input_Text) | set(test_df.Input_Text)
    cache_ok = False
    if os.path.exists(CFG['bt_cache']):
        aug_df = pd.read_csv(CFG['bt_cache'], keep_default_na=False, na_values=[''])
        # cache is valid only if it was built from this exact training split
        cache_ok = aug_df.src_text.isin(set(train_df.Input_Text)).all() and not aug_df.Input_Text.isin(
            set(val_df.Input_Text) | set(test_df.Input_Text)).any()
        print('loaded BT cache:', len(aug_df), 'rows | valid:', cache_ok)
    if not cache_ok:
        t0 = time.time()
        aug_df = back_translate(train_df, all_texts)
        aug_df.to_csv(CFG['bt_cache'], index=False)
        print(f'back-translation done in {(time.time()-t0)/60:.1f} min -> {CFG["bt_cache"]}')
    aug_df = encode_labels(aug_df)      # re-derive y1..y5 from the text labels (safe after CSV round-trip)
    aug_df['sw'] = 1.0; aug_df['is_aug'] = 1
    print(aug_df.groupby('Level1_Class').size().to_string())
    for _, r in aug_df.sample(min(5, len(aug_df)), random_state=SEED).iterrows():
        print(f'\n[{r.Level1_Class}/{r.Level3_Category}] {r.src_text}\n  -> ({r.pivot}) {r.Input_Text}')

train_full = pd.concat([train_df, aug_df[train_df.columns]], ignore_index=True)
print(f'\ntraining rows: {len(train_df)} original + {len(aug_df)} back-translated = {len(train_full)}')

loaded BT cache: 2782 rows | valid: True
Level1_Class
HATE        2043
NON-HATE     739

[NON-HATE/Nil] என்ன ஆக மாமா இருந்தா என்ன
  -> (eng_Latn) என்ன மாமா நடந்தது என்ன

[HATE/IMPLICIT] நேத்து பிளு ஃபில்ம் ல நடிச்சவங்க இன்னைக்கு அம்மா aaitanga
  -> (hin_Deva) நேற்று ப்ளூ படத்தில் நடித்து இன்று அம்மா aaitanga

[NON-HATE/Nil] இது போன்று நடப்பதை தடுக்க ஆன்லைன் வகுப்புகளை இனி வீடியோ பதிவு செய்ய வேண்டும்
  -> (eng_Latn) இதைத் தடுக்க, ஆன்லைன் வகுப்புகள் வீடியோவாக எடுக்கப்பட வேண்டும்.

[NON-HATE/Nil] கார்த்தி நா பாத்த. அவன .முகத்த மட்டும் பாத்துர:
  -> (eng_Latn) கார்த்தி நா பாத்தா. அவர் என்னைப் பார்த்தார்.

[HATE/Nil] பார்ப்பனியம் தமிழ் நாட்டில் கடைவிரிக்க முதலீடாக இருந்தது ஜாதி. ; ; பாப்பானுங்க தான் ஜாதியை உண்டாக்கியது.
  -> (hin_Deva) பார்பனியம் என்பது தமிழில் ஒரு முதலீடாகும், இது ஜாதிகளை பராமரிப்பதற்காக செய்யப்பட்டது; ஜாதிகளை உருவாக்கியது பாகிஸ்தான்.

training rows: 2968 original + 2782 back-translated = 5750


## Hierarchical multi-task MuRIL
Shared encoder → `[CLS ; mean-pool]`. L1/L2/L3 heads read the text representation; the **L4 head is conditioned on the L2 and L3 probability vectors** and the **L5 head on L2**, so lower levels see what the upper levels decided. At inference, L4 is additionally restricted to subcategories that are valid for the predicted (target, category) pair.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
_lens = [len(x) for x in tokenizer(train_df.Input_Text.tolist(), add_special_tokens=True)['input_ids']]
print(f'token length: median {np.median(_lens):.0f}, 95th pct {np.percentile(_lens, 95):.0f}, '
      f'truncated at {CFG["max_len"]}: {np.mean(np.array(_lens) > CFG["max_len"])*100:.1f}%')

class HateDS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.texts = d.Input_Text.tolist()
        self.y = d[[f'y{k}' for k in range(1, 6)]].values.astype(np.int64)
        self.w = d['sw'].values.astype(np.float32)
    def __len__(self): return len(self.texts)
    def __getitem__(self, i): return self.texts[i], self.y[i], self.w[i]

def collate(batch):
    texts, y, w = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True, max_length=CFG['max_len'], return_tensors='pt')
    enc = dict(enc)
    enc['labels'] = torch.tensor(np.stack(y)); enc['sw'] = torch.tensor(np.array(w))
    return enc

def make_loader(d, shuffle, bs):
    return torch.utils.data.DataLoader(HateDS(d), batch_size=bs, shuffle=shuffle, collate_fn=collate, num_workers=0)

class HierMuril(nn.Module):
    def __init__(self, name, num, dropout=0.2):
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        H = self.enc.config.hidden_size
        D = 2 * H
        def head(i, o):
            return nn.Sequential(nn.Dropout(dropout), nn.Linear(i, H // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(H // 2, o))
        self.h1 = head(D, num[1])
        self.h2 = head(D, num[2])
        self.h3 = head(D, num[3])
        self.h4 = head(D + num[2] + num[3], num[4])   # subcategory | target, category
        self.h5 = head(D + num[2], num[5])            # bias | target

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids).last_hidden_state
        m = attention_mask.unsqueeze(-1).to(out.dtype)
        mean = (out * m).sum(1) / m.sum(1).clamp(min=1.0)
        h = torch.cat([out[:, 0], mean], dim=-1)
        z1, z2, z3 = self.h1(h), self.h2(h), self.h3(h)
        p2, p3 = F.softmax(z2, -1).detach(), F.softmax(z3, -1).detach()
        z4 = self.h4(torch.cat([h, p2, p3], -1))
        z5 = self.h5(torch.cat([h, p2], -1))
        return [z1, z2, z3, z4, z5]

def class_weights(d):
    W = {}
    for k in range(1, 6):
        y = d[f'y{k}'].values; y = y[y >= 0]
        cnt = np.bincount(y, minlength=NUM[k]).astype(float)
        w = np.where(cnt > 0, (cnt.sum() / (NUM[k] * np.maximum(cnt, 1))) ** CFG['cw_power'], 0.0)
        w = np.clip(w / w[cnt > 0].mean(), 0.2, 5.0)
        W[k] = torch.tensor(w, dtype=torch.float, device=DEVICE)
    return W

def compute_loss(logits, labels, sw, W):
    total = 0.0
    for k in range(1, 6):
        y = labels[:, k - 1]; valid = y != -100
        if not valid.any():
            continue
        lg, yy, w = logits[k - 1][valid].float(), y[valid], sw[valid]
        ce = F.cross_entropy(lg, yy, weight=W[k], reduction='none', label_smoothing=CFG['label_smoothing'])
        if CFG['focal_gamma'] > 0:
            pt = torch.exp(-F.cross_entropy(lg, yy, reduction='none'))
            ce = ce * (1 - pt) ** CFG['focal_gamma']
        total = total + CFG['loss_weights'][k] * (ce * w).sum() / w.sum()
    return total

def forward_batch(model, b):
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        return model(b['input_ids'], b['attention_mask'], b.get('token_type_ids'))

@torch.no_grad()
def predict_probs(model, d):
    model.eval()
    outs = [[] for _ in range(5)]
    for b in make_loader(d, False, CFG['eval_batch_size']):
        b = {k: v.to(DEVICE) for k, v in b.items()}
        for k, z in enumerate(forward_batch(model, b)):
            outs[k].append(F.softmax(z.float(), -1).cpu().numpy())
    return [np.concatenate(o) for o in outs]

def quick_scores(P, d):
    """Validation score for early stopping: L1 macro-F1 + macro-F1 of L2-L5 on labelled HATE rows."""
    y1 = d.y1.values; p1 = P[0].argmax(1)
    s = {'L1_macroF1': f1_score(y1, p1, average='macro'), 'L1_hateF2': fbeta_score(y1, p1, beta=2)}
    for k in range(2, 6):
        y = d[f'y{k}'].values; m = (y1 == 1) & (y >= 0)
        s[f'L{k}_mF1'] = f1_score(y[m], P[k - 1][m].argmax(1), average='macro', labels=np.unique(y[m]), zero_division=0)
    s['score'] = 0.4 * s['L1_macroF1'] + 0.6 * np.mean([s[f'L{k}_mF1'] for k in range(2, 6)])
    return s

def train_stage(model, train_d, val_d, epochs, enc_lr, head_lr, tag, verbose=True):
    W = class_weights(train_d)
    loader = make_loader(train_d, True, CFG['batch_size'])
    nd = ('bias', 'LayerNorm.weight', 'layer_norm.weight')
    enc_p = list(model.enc.named_parameters())
    head_p = [p for n, p in model.named_parameters() if not n.startswith('enc.')]
    opt = torch.optim.AdamW([
        {'params': [p for n, p in enc_p if not any(x in n for x in nd)], 'lr': enc_lr, 'weight_decay': CFG['weight_decay']},
        {'params': [p for n, p in enc_p if any(x in n for x in nd)], 'lr': enc_lr, 'weight_decay': 0.0},
        {'params': head_p, 'lr': head_lr, 'weight_decay': CFG['weight_decay']},
    ])
    steps = len(loader) * epochs
    sch = get_linear_schedule_with_warmup(opt, int(steps * CFG['warmup_ratio']), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best, best_state, bad, hist = -1.0, None, 0, []
    for ep in range(1, epochs + 1):
        model.train(); run, t0 = 0.0, time.time()
        for b in tqdm(loader, desc=f'{tag} epoch {ep}', leave=False):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = compute_loss(forward_batch(model, b), b['labels'], b['sw'], W)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
            run += loss.item()
        s = quick_scores(predict_probs(model, val_d), val_d)
        s.update(epoch=ep, train_loss=run / len(loader)); hist.append(s)
        if verbose: print(f"[{tag}] ep {ep:2d} | loss {s['train_loss']:.4f} | val L1 mF1 {s['L1_macroF1']:.3f} F2 {s['L1_hateF2']:.3f} | "
              f"L2 {s['L2_mF1']:.3f} L3 {s['L3_mF1']:.3f} L4 {s['L4_mF1']:.3f} L5 {s['L5_mF1']:.3f} | "
              f"score {s['score']:.4f} | {time.time()-t0:.0f}s")
        if s['score'] > best + 1e-4:
            best, bad = s['score'], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG['patience']:
                if verbose: print(f'[{tag}] early stop')
                break
    model.load_state_dict(best_state)
    print(f'[{tag}] best val score {best:.4f}')
    return best, best_state, pd.DataFrame(hist)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

token length: median 20, 95th pct 68, truncated at 128: 1.0%


## F2 threshold tuning, hierarchical decoding, evaluation
* **L1**: threshold on P(HATE) that maximises F2 on validation (recall weighted 4× precision).
* **L2–L5**: per-class log-probability offsets found by coordinate ascent to maximise **macro-F2** on validation HATE rows → raises recall of minority classes.
* **L4**: offsets are tuned on the abuse-type head; the full subcategory is decoded jointly from type + target + explicit/implicit, restricted to combinations seen in training.
* Test metrics: **oracle** (sub-levels scored on gold-HATE rows) and **end-to-end** (a hate post missed at L1 counts as wrong at every lower level).

In [ ]:
def tune_offsets(logp, y, beta, mask=None, grid=np.linspace(-1.5, 1.5, 31), rounds=2):
    K = logp.shape[1]; b = np.zeros(K); labels = np.unique(y)
    base = logp if mask is None else np.where(mask, logp, -1e9)
    score = lambda b: fbeta_score(y, (base + b).argmax(1), beta=beta, average='macro', labels=labels, zero_division=0)
    best = score(b)
    for _ in range(rounds):
        for c in range(K):
            for g in grid:
                bb = b.copy(); bb[c] = g; s = score(bb)
                if s > best + 1e-9:
                    best, b = s, bb
    return b, best

def l4_mask(p2, p3):
    m = COMPAT[p2, p3].copy()
    m[~m.any(1)] = True          # unseen (target, category) pair -> no restriction
    return m

def tune_on_val(Pv, dv, beta):
    y1, ph = dv.y1.values, Pv[0][:, 1]
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    f = [fbeta_score(y1, (ph >= t).astype(int), beta=beta, zero_division=0) for t in grid]
    T = {'thr': float(grid[int(np.argmax(f))]), 'off': {}, 'val_score': {'L1_F2': float(max(f))}}
    for k in (2, 3, 5):
        y = dv[f'y{k}'].values; m = (y1 == 1) & (y >= 0)
        T['off'][k], T['val_score'][f'L{k}_macroF2'] = tune_offsets(np.log(Pv[k - 1][m] + 1e-9), y[m], beta)
    # L4: offsets on the abuse-type head (types are shared across targets -> more stable to tune)
    y = dv.y4.values; m = (y1 == 1) & (y >= 0)
    T['off'][4], T['val_score']['L4type_macroF2'] = tune_offsets(np.log(Pv[3][m] + 1e-9), y[m], beta)
    return T

def l4_full_scores(P, off4, p2, p3, constrain):
    """Score every full L4 subcategory = log P(type) + log P(its target) + log P(its explicit/implicit).
    Sarcasm/None/Other carry no target/category, so they use the predicted L2/L3 scores."""
    n = len(p2); rows = np.arange(n)
    lt = np.log(P[3] + 1e-9) + off4
    l2, l3 = np.log(P[1] + 1e-9), np.log(P[2] + 1e-9)
    s = lt[:, TYPE_OF]
    s = s + np.where(T2_OF >= 0, l2[:, np.maximum(T2_OF, 0)], l2[rows, p2][:, None])
    s = s + np.where(T3_OF >= 0, l3[:, np.maximum(T3_OF, 0)], l3[rows, p3][:, None])
    if constrain:
        s = np.where(l4_mask(p2, p3), s, -1e9)
    return s

def decode(P, T, tuned=True, constrain=True):
    """pred[4] = full L4 subcategory index (into FULL4); pred['4t'] = its abuse type."""
    pred = {1: (P[0][:, 1] >= (T['thr'] if tuned else 0.5)).astype(int)}
    for k in (2, 3, 5):
        pred[k] = (np.log(P[k - 1] + 1e-9) + (T['off'][k] if tuned else 0)).argmax(1)
    pred[4] = l4_full_scores(P, T['off'][4] if tuned else 0, pred[2], pred[3], constrain).argmax(1)
    pred['4t'] = TYPE_OF[pred[4]]
    return pred

def evaluate(P, d, T, tuned, constrain, verbose=False):
    pred = decode(P, T, tuned, constrain); y1 = d.y1.values; p1 = pred[1]
    r = {'L1_acc': accuracy_score(y1, p1), 'L1_macroF1': f1_score(y1, p1, average='macro'),
         'L1_hate_P': precision_score(y1, p1, zero_division=0), 'L1_hate_R': recall_score(y1, p1),
         'L1_hate_F1': f1_score(y1, p1), 'L1_hate_F2': fbeta_score(y1, p1, beta=2)}
    for cat in ('IMPLICIT', 'EXPLICIT'):
        m = (y1 == 1) & (d.Level3_Category.values == cat)
        r[f'L1_recall_{cat.lower()}'] = p1[m].mean() if m.any() else np.nan
    # (metric name, gold column, prediction key, class names); L4 = full subcategory, L4type = abuse type
    for name, col, pk, names in [('L2', 'y2', 2, CLASSES[2]), ('L3', 'y3', 3, CLASSES[3]), ('L4', 'y4f', 4, FULL4),
                                 ('L4type', 'y4', '4t', CLASSES[4]), ('L5', 'y5', 5, CLASSES[5])]:
        y = d[col].values; m = (y1 == 1) & (y >= 0); labels = np.unique(y[m])
        yo = pred[pk][m]; ye = np.where(p1[m] == 1, yo, -1)       # -1 = missed at L1 -> wrong
        r[f'{name}_oracle_mF1'] = f1_score(y[m], yo, average='macro', labels=labels, zero_division=0)
        r[f'{name}_oracle_mF2'] = fbeta_score(y[m], yo, beta=2, average='macro', labels=labels, zero_division=0)
        r[f'{name}_oracle_mRecall'] = recall_score(y[m], yo, average='macro', labels=labels, zero_division=0)
        r[f'{name}_oracle_acc'] = accuracy_score(y[m], yo)
        r[f'{name}_e2e_mF1'] = f1_score(y[m], ye, average='macro', labels=labels, zero_division=0)
        if verbose:
            print(f'\n===== {name} (oracle: gold-HATE test rows, n={m.sum()}) =====')
            print(classification_report(y[m], yo, labels=labels, target_names=[names[i] for i in labels], zero_division=0))
    ph = p1 == 1
    r['hier_consistency'] = COMPAT[pred[2][ph], pred[3][ph], pred[4][ph]].mean() if ph.any() else np.nan
    if verbose:
        print(f"\n===== Level1 (threshold {T['thr'] if tuned else 0.5}) =====")
        print(classification_report(y1, p1, target_names=CLASSES[1], digits=3))
        print(confusion_matrix(y1, p1))
    return r

RESULTS = {}
def full_eval(model, tag):
    Pv, Pt = predict_probs(model, val_df), predict_probs(model, test_df)
    T = tune_on_val(Pv, val_df, CFG['beta'])
    RESULTS[f'{tag} | argmax'] = evaluate(Pt, test_df, T, tuned=False, constrain=False)
    RESULTS[f'{tag} | argmax + hier-decode'] = evaluate(Pt, test_df, T, tuned=False, constrain=True)
    if CFG['use_thresholds']:
        RESULTS[f'{tag} | F2-thresholds + hier-decode'] = evaluate(Pt, test_df, T, tuned=True, constrain=True)
    print(f"[{tag}] tuned L1 threshold = {T['thr']:.2f} | val tuning scores: "
          + ', '.join(f'{k} {v:.3f}' for k, v in T['val_score'].items()))
    return T, Pt

SHOW = ['L1_macroF1', 'L1_hate_R', 'L1_hate_F2', 'L1_recall_implicit', 'L1_recall_explicit',
        'L2_oracle_mF1', 'L3_oracle_mF1', 'L4_oracle_mF1', 'L4type_oracle_mF1', 'L5_oracle_mF1',
        'L2_oracle_mRecall', 'L4_oracle_mRecall', 'L5_oracle_mRecall',
        'L2_e2e_mF1', 'L4_e2e_mF1', 'hier_consistency']
def show_results():
    print(pd.DataFrame(RESULTS).T[SHOW].round(3).to_string())

## Stage 1 — train on original + back-translated data

In [ ]:
set_seed(SEED)
model = HierMuril(CFG['model_name'], NUM, CFG['head_dropout']).to(DEVICE)
best1, stage1_state, hist1 = train_stage(model, train_full, val_df, CFG['stage1_epochs'], CFG['enc_lr'], CFG['head_lr'], 'stage1')
tag1 = 'S1' + (' +BT' if CFG['use_bt'] else '')
T1, Pt1 = full_eval(model, tag1)
show_results()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

stage1 epoch 1:   0%|          | 0/360 [00:00<?, ?it/s]

... (15 outputs left out)

stage1 epoch 9:   0%|          | 0/360 [00:00<?, ?it/s]

[stage1] ep  9 | loss 0.4483 | val L1 mF1 0.802 F2 0.830 | L2 0.633 L3 0.703 L4 0.176 L5 0.444 | score 0.6143 | 63s


stage1 epoch 10:   0%|          | 0/360 [00:00<?, ?it/s]

[stage1] ep 10 | loss 0.3839 | val L1 mF1 0.797 F2 0.828 | L2 0.640 L3 0.698 L4 0.174 L5 0.455 | score 0.6139 | 64s
[stage1] best val score 0.6157
[S1 +BT] tuned L1 threshold = 0.06 | val tuning scores: L1_F2 0.871, L2_macroF2 0.656, L3_macroF2 0.717, L5_macroF2 0.480, L4type_macroF2 0.249
                                      L1_macroF1  L1_hate_R  L1_hate_F2  L1_recall_implicit  L1_recall_explicit  L2_oracle_mF1  L3_oracle_mF1  L4_oracle_mF1  L4type_oracle_mF1  L5_oracle_mF1  L2_oracle_mRecall  L4_oracle_mRecall  L5_oracle_mRecall  L2_e2e_mF1  L4_e2e_mF1  hier_consistency
S1 +BT | argmax                            0.798      0.817       0.814               0.726               0.923          0.628          0.742          0.171              0.195          0.375              0.629              0.183              0.393       0.581       0.157             0.988
S1 +BT | argmax + hier-decode              0.798      0.817       0.814               0.726               0.923          0.628   

## Hard-negative mining (out-of-fold) → Stage 2
Mining with the stage-1 model is useless: it has already memorised the training set, so nothing looks "hard". Instead, the training originals are split into `hnm_folds` folds; for each fold a fresh MuRIL is trained for `hnm_fold_epochs` epochs on the other folds (their back-translations included, the held-out rows' paraphrases excluded) and scores the held-out rows. Every training row therefore gets an **out-of-fold P(HATE)** from a model that never saw it.
* **Hard negatives**: NON-HATE rows with the highest OOF P(HATE) (top `hnm_top_frac` + every OOF false positive)
* **Hard positives**: HATE rows with the lowest OOF P(HATE) (top `hnm_top_frac` + every OOF false negative, mostly implicit hate)
* Their weight, and their paraphrases' weight, is raised to `hnm_weight`, and the stage-1 model is fine-tuned at a lower LR. Stage 2 is kept only if it beats stage 1 on validation (its test row is still shown in the table for the ablation).
* Very confident disagreements (NON-HATE with OOF P ≥ 0.9, HATE with OOF P ≤ 0.1) are likely **label errors**; they are saved to `label_review.csv` for manual checking.

In [ ]:
from sklearn.model_selection import StratifiedKFold

final_tag, best_final = tag1, best1
T_final, Pt_final = T1, Pt1
if CFG['use_hnm']:
    orig = train_full[train_full.is_aug == 0]
    oof = np.zeros(len(orig))
    skf = StratifiedKFold(n_splits=CFG['hnm_folds'], shuffle=True, random_state=SEED)
    for fi, (tr_i, ho_i) in enumerate(skf.split(orig, orig.strat)):
        ho_texts = set(orig.Input_Text.iloc[ho_i])
        fold_train = train_full[~train_full.Input_Text.isin(ho_texts) & ~train_full.src_text.isin(ho_texts)]
        set_seed(SEED + 100 + fi)
        fm = HierMuril(CFG['model_name'], NUM, CFG['head_dropout']).to(DEVICE)
        fb, _, _ = train_stage(fm, fold_train, val_df, CFG['hnm_fold_epochs'], CFG['enc_lr'], CFG['head_lr'],
                               f'oof-fold{fi + 1}', verbose=False)
        oof[ho_i] = predict_probs(fm, orig.iloc[ho_i])[0][:, 1]
        y_ho = orig.y1.values[ho_i]
        print(f'fold {fi + 1}/{CFG["hnm_folds"]}: trained on {len(fold_train)} rows, val score {fb:.3f}, '
              f'held-out L1 macro-F1 {f1_score(y_ho, (oof[ho_i] >= 0.5).astype(int), average="macro"):.3f}')
        del fm; gc.collect(); torch.cuda.empty_cache()

    ph, y1 = oof, orig.y1.values
    f = CFG['hnm_top_frac']
    hard_neg = (y1 == 0) & ((ph >= np.quantile(ph[y1 == 0], 1 - f)) | (ph >= 0.5))
    hard_pos = (y1 == 1) & ((ph <= np.quantile(ph[y1 == 1], f)) | (ph < 0.5))
    hard_idx = orig.index[hard_neg | hard_pos]
    print(f'\nOOF L1 macro-F1 on training originals: {f1_score(y1, (ph >= 0.5).astype(int), average="macro"):.3f}')
    print(f'hard negatives: {hard_neg.sum()} (mean OOF P(hate) {ph[hard_neg].mean():.3f}) | '
          f'hard positives: {hard_pos.sum()} (mean OOF P(hate) {ph[hard_pos].mean():.3f}, '
          f'{(orig.Level3_Category.values[hard_pos] == "IMPLICIT").mean()*100:.0f}% implicit)')

    print('\nTop hard negatives (NON-HATE that look like hate):')
    for i in np.argsort(-np.where(y1 == 0, ph, -1))[:8]:
        print(f'  P={ph[i]:.2f}  {orig.Input_Text.iloc[i][:120]}')
    print('\nTop hard positives (HATE that look harmless):')
    for i in np.argsort(np.where(y1 == 1, ph, 2))[:8]:
        print(f'  P={ph[i]:.2f}  [{orig.Level3_Category.iloc[i]}] {orig.Input_Text.iloc[i][:120]}')

    # likely annotation errors -> file for manual review
    suspect = ((y1 == 0) & (ph >= 0.9)) | ((y1 == 1) & (ph <= 0.1))
    review = orig.loc[suspect, ['Input_Text'] + LEVELS].copy()
    review.insert(1, 'OOF_P_HATE', ph[suspect].round(3))
    review.sort_values('OOF_P_HATE', ascending=False).to_csv('label_review.csv', index=False)
    print(f'\n{suspect.sum()} likely label errors saved to label_review.csv (check them manually)')

    train_hnm = train_full.copy()
    train_hnm.loc[hard_idx, 'sw'] = CFG['hnm_weight']
    hard_texts = set(orig.loc[hard_idx, 'Input_Text'])
    train_hnm.loc[train_hnm.src_text.isin(hard_texts), 'sw'] = CFG['hnm_weight']   # their paraphrases too
    if CFG['hnm_drop_suspects']:
        bad = set(orig.Input_Text[suspect])
        train_hnm = train_hnm[~train_hnm.Input_Text.isin(bad) & ~train_hnm.src_text.isin(bad)]
        print(f'dropped {suspect.sum()} suspect rows (+ their paraphrases) from stage 2')
    print(f"up-weighted rows (incl. paraphrases): {(train_hnm.sw > 1).sum()} / {len(train_hnm)}")

    best2, stage2_state, hist2 = train_stage(model, train_hnm, val_df, CFG['stage2_epochs'],
                                             CFG['stage2_enc_lr'], CFG['stage2_head_lr'], 'stage2')
    T2, Pt2 = full_eval(model, tag1 + ' +HNM')
    if best2 > best1:
        final_tag, best_final, T_final, Pt_final = tag1 + ' +HNM', best2, T2, Pt2
    else:
        print('stage 2 did not improve validation -> keeping stage-1 model')
        model.load_state_dict(stage1_state)

print(f'\nFINAL model: {final_tag} (val score {best_final:.4f})')
show_results()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

oof-fold1 epoch 1:   0%|          | 0/240 [00:00<?, ?it/s]

oof-fold1 epoch 2:   0%|          | 0/240 [00:00<?, ?it/s]

oof-fold1 epoch 3:   0%|          | 0/240 [00:00<?, ?it/s]

[oof-fold1] best val score 0.5571
fold 1/3: trained on 3827 rows, val score 0.557, held-out L1 macro-F1 0.795


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (13 outputs left out)

stage2 epoch 3:   0%|          | 0/360 [00:00<?, ?it/s]

[stage2] ep  3 | loss 0.3156 | val L1 mF1 0.795 F2 0.842 | L2 0.637 L3 0.713 L4 0.165 L5 0.402 | score 0.6055 | 63s


stage2 epoch 4:   0%|          | 0/360 [00:00<?, ?it/s]

[stage2] ep  4 | loss 0.2857 | val L1 mF1 0.802 F2 0.840 | L2 0.634 L3 0.720 L4 0.176 L5 0.406 | score 0.6110 | 63s
[stage2] best val score 0.6118
[S1 +BT +HNM] tuned L1 threshold = 0.11 | val tuning scores: L1_F2 0.869, L2_macroF2 0.648, L3_macroF2 0.725, L5_macroF2 0.481, L4type_macroF2 0.259
stage 2 did not improve validation -> keeping stage-1 model

FINAL model: S1 +BT (val score 0.6157)
                                           L1_macroF1  L1_hate_R  L1_hate_F2  L1_recall_implicit  L1_recall_explicit  L2_oracle_mF1  L3_oracle_mF1  L4_oracle_mF1  L4type_oracle_mF1  L5_oracle_mF1  L2_oracle_mRecall  L4_oracle_mRecall  L5_oracle_mRecall  L2_e2e_mF1  L4_e2e_mF1  hier_consistency
S1 +BT | argmax                                 0.798      0.817       0.814               0.726               0.923          0.628          0.742          0.171              0.195          0.375              0.629              0.183              0.393       0.581       0.157             0.988
S1 +BT | argma

In [ ]:
# ---------- detailed test report for the final model ----------
final_metrics = evaluate(Pt_final, test_df, T_final, tuned=CFG['use_thresholds'], constrain=True, verbose=True)
print('\nL1 recall on implicit hate: %.3f | explicit hate: %.3f' % (final_metrics['L1_recall_implicit'], final_metrics['L1_recall_explicit']))


===== L2 (oracle: gold-HATE test rows, n=327) =====
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.45      0.62      0.52        21
   Hate against Men       0.64      0.64      0.64        92
 Hate against Women       0.78      0.78      0.78       153
 No Target- Profane       0.54      0.46      0.50        61

           accuracy                           0.67       327
          macro avg       0.60      0.63      0.61       327
       weighted avg       0.67      0.67      0.67       327


===== L3 (oracle: gold-HATE test rows, n=330) =====
              precision    recall  f1-score   support

    EXPLICIT       0.74      0.66      0.70       155
    IMPLICIT       0.72      0.79      0.76       175

    accuracy                           0.73       330
   macro avg       0.73      0.73      0.73       330
weighted avg       0.73      0.73      0.73       330


===== L4 (oracle: gold-HATE test rows, n=309) =====
                       

In [ ]:
# ---------- save ----------
out = CFG['out_dir']
os.makedirs(out, exist_ok=True)
torch.save(model.state_dict(), f'{out}/model.pt')
tokenizer.save_pretrained(out)
meta = {
    'cfg': {k: v for k, v in CFG.items()},
    'classes': {str(k): v for k, v in CLASSES.items()},
    'num': {str(k): v for k, v in NUM.items()},
    'thresholds': {'thr': T_final['thr'], 'off': {str(k): np.asarray(v).tolist() for k, v in T_final['off'].items()}},
    'compat': COMPAT.astype(int).tolist(),
    'full4': FULL4, 'type_of': TYPE_OF.tolist(), 't2_of': T2_OF.tolist(), 't3_of': T3_OF.tolist(),
    'final_model': final_tag,
}
with open(f'{out}/meta.json', 'w', encoding='utf-8') as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=1, default=str)
pd.DataFrame(RESULTS).T.to_csv(f'{out}/test_results.csv')
print('saved to', out, os.listdir(out))
# optional: download as zip
# shutil.make_archive(out, 'zip', out); from google.colab import files; files.download(out + '.zip')

saved to tamil_hate_hier_model ['model.pt', 'tokenizer.json', 'meta.json', 'tokenizer_config.json', 'test_results.csv']


In [ ]:
# ---------- reload (for a fresh session) ----------
def load_trained(out_dir):
    global CLASSES, NUM, COMPAT, FULL4, NUM_FULL4, TYPE_OF, T2_OF, T3_OF, tokenizer
    with open(f'{out_dir}/meta.json', encoding='utf-8') as fh:
        meta = json.load(fh)
    CLASSES = {int(k): v for k, v in meta['classes'].items()}
    NUM = {int(k): v for k, v in meta['num'].items()}
    COMPAT = np.array(meta['compat'], dtype=bool)
    FULL4, NUM_FULL4 = meta['full4'], len(meta['full4'])
    TYPE_OF, T2_OF, T3_OF = (np.array(meta[k]) for k in ('type_of', 't2_of', 't3_of'))
    tokenizer = AutoTokenizer.from_pretrained(out_dir)
    m = HierMuril(meta['cfg']['model_name'], NUM, meta['cfg']['head_dropout'])
    m.load_state_dict(torch.load(f'{out_dir}/model.pt', map_location='cpu'))
    T = {'thr': meta['thresholds']['thr'], 'off': {int(k): np.array(v) for k, v in meta['thresholds']['off'].items()}}
    return m.to(DEVICE).eval(), T
# model, T_final = load_trained(CFG['out_dir'])

In [ ]:
# ---------- inference ----------
def predict(texts, mdl=None, T=None):
    mdl = mdl or model
    T = T or T_final
    single = isinstance(texts, str)
    texts = [texts] if single else list(texts)
    d = pd.DataFrame({'Input_Text': [normalize_text(t) for t in texts], 'sw': 1.0})
    for k in range(1, 6):
        d[f'y{k}'] = -100
    P = predict_probs(mdl, d)
    pred = decode(P, T, tuned=CFG['use_thresholds'], constrain=True)
    res = []
    for i, t in enumerate(texts):
        hate = bool(pred[1][i])
        r = {'text': t, 'P(HATE)': round(float(P[0][i, 1]), 3), 'Level1_Class': 'HATE' if hate else 'NON-HATE'}
        for k in range(2, 6):
            names = FULL4 if k == 4 else CLASSES[k]
            lab = names[pred[k][i]] if hate else 'Nil'
            r[COLS[k]] = 'Nil' if lab in ('None', 'No-Bias') else lab
            if hate:   # confidence of the head that made the decision (abuse-type head for L4)
                ci = pred['4t'][i] if k == 4 else pred[k][i]
                r[f'{COLS[k]}_conf'] = round(float(P[k - 1][i, ci]), 3)
        res.append(r)
    return res[0] if single else res

predict("இந்த காலத்து பொண்ணுங்க டிரஸ் பாத்தாலே தெரியுது எப்படிப்பட்டவங்கனு")

{'text': 'இந்த காலத்து பொண்ணுங்க டிரஸ் பாத்தாலே தெரியுது எப்படிப்பட்டவங்கனு',
 'P(HATE)': 0.985,
 'Level1_Class': 'HATE',
 'Level2_Target': 'Hate against Women',
 'Level2_Target_conf': 0.957,
 'Level3_Category': 'IMPLICIT',
 'Level3_Category_conf': 0.999,
 'Level4_Subcategory': 'ImplicitWomen-Moral policing',
 'Level4_Subcategory_conf': 0.439,
 'Level5_Bias': 'Nil',
 'Level5_Bias_conf': 0.945}

In [ ]:
# a few random test rows: gold vs predicted
for _, r in test_df.sample(5, random_state=1).iterrows():
    p = predict(r.Input_Text)
    print(r.Input_Text[:100])
    print('  gold:', [r[c] for c in LEVELS])
    print('  pred:', [p[c] for c in LEVELS], 'P(HATE)=', p['P(HATE)'], '\n')

வாயில நுரை தல்லுது ஆனாலும் பேச்சு குறையலையே,,,இவளுக்கு
  gold: ['HATE', 'Hate against Women', 'IMPLICIT', 'Sarcasm', 'No-Bias']
  pred: ['HATE', 'Hate against Women', 'IMPLICIT', 'ImplicitWomen-Slut-shaming', 'Nil'] P(HATE)= 0.902 

களாச்சரத்தை சீரழிக்கும் நிகழ்ச்சி தமிழா தமிழா
  gold: ['NON-HATE', 'Nil', 'Nil', 'Nil', 'No-Bias']
  pred: ['HATE', 'No Target- Profane', 'IMPLICIT', 'Nil', 'Nil'] P(HATE)= 0.988 

இந்த பிராமணன் வைரமுத்து மீது உள்ள பாலியல் புகார் பற்றி பேச வேண்டும்.
  gold: ['NON-HATE', 'Nil', 'Nil', 'Nil', 'No-Bias']
  pred: ['HATE', 'Hate against Men', 'IMPLICIT', 'ImplicitMen-Character assassination', 'Nil'] P(HATE)= 0.366 

Tamil selvan பாடே ஓழ் வாங்கவே இங்க வாறீயா😁😁😁
  gold: ['HATE', 'Hate against Men', 'EXPLICIT', 'ExplicitMen-Verbal Abusive', 'No-Bias']
  pred: ['HATE', 'Hate against LGBTQ+', 'EXPLICIT', 'ExplicitLGBTQ-Other', 'Gender'] P(HATE)= 0.064 

எட்சை baadu raja nee intha வார்த்தைய பப்ளிக் பேசுட பார்ப்போம் baadu
  gold: ['HATE', 'Hate against Men', 'EXPLICIT'

In [ ]:
print(pd.DataFrame(RESULTS).T[['L1_acc','L2_oracle_acc','L3_oracle_acc','L4_oracle_acc','L5_oracle_acc']].round(3))

                                           L1_acc  L2_oracle_acc  \
S1 +BT | argmax                             0.799          0.682   
S1 +BT | argmax + hier-decode               0.799          0.682   
S1 +BT | F2-thresholds + hier-decode        0.767          0.673   
S1 +BT +HNM | argmax                        0.794          0.682   
S1 +BT +HNM | argmax + hier-decode          0.794          0.682   
S1 +BT +HNM | F2-thresholds + hier-decode   0.777          0.679   

                                           L3_oracle_acc  L4_oracle_acc  \
S1 +BT | argmax                                    0.748          0.282   
S1 +BT | argmax + hier-decode                      0.748          0.288   
S1 +BT | F2-thresholds + hier-decode               0.730          0.272   
S1 +BT +HNM | argmax                               0.742          0.285   
S1 +BT +HNM | argmax + hier-decode                 0.742          0.288   
S1 +BT +HNM | F2-thresholds + hier-decode          0.727          0.278  

In [ ]:
# ---------- inference ----------
def predict(texts, mdl=None, T=None):
    mdl = mdl or model
    T = T or T_final
    single = isinstance(texts, str)
    texts = [texts] if single else list(texts)
    d = pd.DataFrame({'Input_Text': [normalize_text(t) for t in texts], 'sw': 1.0})
    for k in range(1, 6):
        d[f'y{k}'] = -100
    P = predict_probs(mdl, d)
    pred = decode(P, T, tuned=CFG['use_thresholds'], constrain=True)
    res = []
    for i, t in enumerate(texts):
        hate = bool(pred[1][i])
        r = {'text': t, 'P(HATE)': round(float(P[0][i, 1]), 3), 'Level1_Class': 'HATE' if hate else 'NON-HATE'}
        for k in range(2, 6):
            names = FULL4 if k == 4 else CLASSES[k]
            lab = names[pred[k][i]] if hate else 'Nil'
            r[COLS[k]] = 'Nil' if lab in ('None', 'No-Bias') else lab
            if hate:   # confidence of the head that made the decision (abuse-type head for L4)
                ci = pred['4t'][i] if k == 4 else pred[k][i]
                r[f'{COLS[k]}_conf'] = round(float(P[k - 1][i, ci]), 3)
        res.append(r)
    return res[0] if single else res


# ---------- test samples ----------
# ---------- test samples ----------
samples = [
    # NON-HATE (expected: NON-HATE)
    "இந்த வீடியோ ரொம்ப அருமையா இருக்கு, நன்றி அண்ணா",
    "நாளைக்கு சென்னையில மழை வருமா?",
    "பெண்கள் படிச்சா தான் நாடு முன்னேறும்",
    "super speech sir, ellarum kandippa paakanum",
    # EXPLICIT HATE (expected: HATE, EXPLICIT)
    "இவன் ஒரு முட்டாள் நாய், வாயை மூடு",
    "இவள மாதிரி கேவலமான பொம்பள யாரும் இல்ல",
    "dei loosu payale, poi sethu po",
    "இவனுங்க எல்லாம் மனுஷங்களே இல்ல, ஊர விட்டு விரட்டணும்",
    # IMPLICIT HATE (expected: HATE, IMPLICIT)
    "பொம்பளைங்க வீட்ல சமையல் பண்ணா போதும், அரசியல் எதுக்கு",
    "இந்த காலத்து பொண்ணுங்க டிரஸ் பாத்தாலே தெரியுது எப்படிப்பட்டவங்கனு",
    "ஆம்பளையா இருந்தா அழக்கூடாது, நீ என்ன பொம்பளையா?",
    "அவங்க ரெண்டு பேரும் சேர்ந்து வாழறது இயற்கைக்கு எதிரானது",
    # SARCASM (tricky)
    "வாவ், என்ன ஒரு அறிவு! இவங்கள மாதிரி ஆளுங்க தான் நாட்டுக்கு தேவை 😂",
    "அம்மணி பேசுறத கேட்டா நோபல் பரிசே குடுக்கலாம் போல 😂",
    # HARD NON-HATE (sensitive topic, but not hate)
    "இந்த நிகழ்ச்சி நம்ம கலாச்சாரத்தை கெடுக்குது",
    "சாதி பாகுபாடு இன்னும் கிராமங்களில் இருக்கு, இதை ஒழிக்கணும்",
]
expected = ['NON-HATE'] * 4 + ['HATE'] * 10 + ['NON-HATE'] * 2

res = pd.DataFrame(predict(samples))
res.insert(1, 'expected', expected)
pd.set_option('display.max_colwidth', 50)
res[['text', 'expected', 'P(HATE)', 'Level1_Class',
     'Level2_Target', 'Level3_Category', 'Level4_Subcategory', 'Level5_Bias']]

,text,expected,P(HATE),Level1_Class,Level2_Target,Level3_Category,Level4_Subcategory,Level5_Bias
0,"இந்த வீடியோ ரொம்ப அருமையா இருக்கு, நன்றி அண்ணா",NON-HATE,0.000,NON-HATE,Nil,Nil,Nil,Nil
1,நாளைக்கு சென்னையில மழை வருமா?,NON-HATE,0.096,HATE,Hate against LGBTQ+,IMPLICIT,Sarcasm,Nil
2,பெண்கள் படிச்சா தான் நாடு முன்னேறும்,NON-HATE,0.086,HATE,Hate against Women,IMPLICIT,ImplicitWomen-Moral policing,Other
3,"super speech sir, ellarum kandippa paakanum",NON-HATE,0.004,NON-HATE,Nil,Nil,Nil,Nil
4,"இவன் ஒரு முட்டாள் நாய், வாயை மூடு",HATE,0.995,HATE,Hate against Men,EXPLICIT,ExplicitMen-Masculinity Attacks,Gender
5,இவள மாதிரி கேவலமான பொம்பள யாரும் இல்ல,HATE,0.998,HATE,Hate against Women,IMPLICIT,ImplicitWomen-Character assassination,Nil
6,"dei loosu payale, poi sethu po",HATE,0.475,HATE,No Target- Profane,IMPLICIT,Nil,Nil
7,"இவனுங்க எல்லாம் மனுஷங்களே இல்ல, ஊர விட்டு விரட...",HATE,0.985,HATE,No Target- Profane,IMPLICIT,Nil,Nil
8,"பொம்பளைங்க வீட்ல சமையல் பண்ணா போதும், அரசியல் ...",HATE,0.989,HATE,Hate against Women,IMPLICIT,ImplicitWomen-Character assassination,Nil
9,இந்த காலத்து பொண்ணுங்க டிரஸ் பாத்தாலே தெரியுது...,HATE,0.985,HATE,Hate against Women,IMPLICIT,ImplicitWomen-Moral policing,Nil
